In [1]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [11]:
stime = 144 # ms
bw = 1342176000 # B per second
seed_list = [1,2,3,4,5]
topologytype = 2
os = 1
nintervals = 8
cm = "prv1"
load = 2
swlist = range(40,121,20)
dringserverlist = [752,1680,2992,4672,6720]

In [4]:
with open("dring_copy_topologyfiles.conf", "w") as f:
    for sw in swlist:
        fromfile = f"{homedir}scalegraphfiles/dring_deg{int(sw*0.8)}_sw{sw}_sn12_i1.edgelist"
        tofile = f"{homedir}evalscaletopologyfiles/dring_{sw}_{int(sw*0.8)}.edgelist"
        f.write(f"cp {fromfile} {tofile}\n")

actually copy

In [ ]:
with open("dring_copy_serverfiles.conf",'w') as f:
    for isw,sw in enumerate(swlist):
        fromfile = f"{homedir}serverfiles/dring_{dringserverlist[isw]}_{sw}_{int(sw*0.8)}"
        tofile = f"{homedir}evalserverfiles/dring_{dringserverlist[isw]}_{sw}_{int(sw*0.8)}.sv"
        f.write(f"cp {fromfile} {tofile}\n")

actually copy

In [ ]:
with open("dringsu2_copy_netpathfiles.conf",'w') as f:
    for isw,sw in enumerate(swlist):
        fromfile = f"{homedir}netpathfiles/netpath_su2_dring_{dringserverlist[isw]}_{sw}_{int(sw*0.8)}"
        tofile = f"{homedir}evalscalenetpathfiles/netpath_dring_{sw}_{int(sw*0.8)}_su2.np"
        f.write(f"cp {fromfile} {tofile}\n")

actually copy

In [16]:
for isw,sw in enumerate(swlist):
    # set parameters
    topologyfile = f"evalscaletopologyfiles/dring_{sw}_{int(sw*0.8)}.edgelist"
    serverfile = f"evalserverfiles/dring_{dringserverlist[isw]}_{sw}_{int(sw*0.8)}.sv"
    npfile = f"evalscalenetpathfiles/netpath_dring_{sw}_{int(sw*0.8)}_su2.np"
    nhosts = dringserverlist[isw]
    nswitches = sw
    k = int(sw*0.8)
    with open(f"{homedir}{topologyfile}", 'r') as f:
        nlines = sum(1 for _ in f)
    nlinks = nlines * 2
    
    
    


    # generate connection_matrices file (1)
    unv1bytes = 0
    unv1file = f'{homedir}rawtrafficfiles/{cm}'
    maxinterval = 0
    with open(unv1file, 'r') as f:
        lines = f.readlines()
        for line in lines:
            tokens = line.split(',')
            # 0,32,31,10500
            # interval,fromserver,toserver,bytes
            unv1bytes += int(tokens[3])
            maxinterval = max(maxinterval, int(tokens[0]))
    print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')






    # generate connection_matrices file (2)
    random.seed(0)
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    mult = totalbytes / unv1bytes
    actualbytes = 0
    cmfile = f'cmfiles/dring_sw{sw}_load{load}.cm'
    with open(cmfile, 'w') as fw:
        with open(unv1file, 'r') as fr:
            lines = fr.readlines()
            iline = 0
            while actualbytes < totalbytes:
                line = lines[iline]
                tokens = line.split(',')
                interval = int(tokens[0])
                fromserver = int(tokens[1])
                toserver = int(tokens[2])
                multbytes = int(tokens[3])

                if fromserver >= nhosts or toserver >= nhosts:
                    iline += 1
                    if iline >= len(lines):
                        iline = 0
                        if mult-1>0:
                            mult = mult-1
                    continue

                if mult >= 1 or (random.random() < mult):
                    multbytes = adjustbytesbymtu(multbytes)
    
                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(multbytes)

                iline += 1
                if iline >= len(lines):
                    iline = 0
                    if mult-1>0:
                        mult = mult-1

                    # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

    print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')

unv1bytes 222240244500, maxinterval 7, fullload 102048325632.0, ratio 0.4591802257128996
load 2%, totalbytes 2040966512.64, unv1bytes 222240244500, mult 0.009183604514257994, actualbytes 2040970500
unv1bytes 222240244500, maxinterval 7, fullload 231928012800.0, ratio 1.043591422074772
load 2%, totalbytes 4638560256.0, unv1bytes 222240244500, mult 0.020871828441495436, actualbytes 4638778500
unv1bytes 222240244500, maxinterval 7, fullload 411285676032.0, ratio 1.8506354551459288
load 2%, totalbytes 8225713520.64, unv1bytes 222240244500, mult 0.037012709102918574, actualbytes 8225715000
unv1bytes 222240244500, maxinterval 7, fullload 643213688832.0, ratio 2.8942268772207007
load 2%, totalbytes 12864273776.64, unv1bytes 222240244500, mult 0.057884537544414014, actualbytes 12864303000
unv1bytes 222240244500, maxinterval 7, fullload 927712051200.0, ratio 4.174365688299088
load 2%, totalbytes 18554241024.0, unv1bytes 222240244500, mult 0.08348731376598174, actualbytes 18554317500


In [24]:
with open('dringsu2_generate_pwfiles.conf', 'w') as fgen:
    with open('dringsu2_copy_pwfiles.conf', 'w') as fcopy:
        for isw,sw in enumerate(swlist):
            # set parameters
            topologyfile = f"evalscaletopologyfiles/dring_{sw}_{int(sw*0.8)}.edgelist"
            serverfile = f"evalserverfiles/dring_{dringserverlist[isw]}_{sw}_{int(sw*0.8)}.sv"
            npfile = f"evalscalenetpathfiles/netpath_dring_{sw}_{int(sw*0.8)}_su2.np"
            nhosts = dringserverlist[isw]
            nswitches = sw
            k = int(sw*0.8)
            with open(f"{homedir}{topologyfile}", 'r') as f:
                nlines = sum(1 for _ in f)
            nlinks = nlines * 2





            # generate pathweight file (1)
            interval_stime = stime / nintervals
            cmfile = f'cmfiles/dring_sw{sw}_load{load}.cm'
            for interval in range(nintervals):
                flowstart = interval_stime * interval
                flowend = interval_stime * (interval + 1)
                varfile = f'{homedir}rawpathweightfiles/pathtraffic_dring_sw{sw}_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{interval}.var'
                qvarfile = f'{homedir}rawpathweightfiles/pathweight_dring_sw{sw}_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{interval}.var'
                fgen.write(f"python3 {homedir}generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")





            # generate pathweight file (2)
            intervaldict = {0:0,1:0,2:1,3:2,4:3,5:4,6:5,7:6} # to:from
            for interval in range(nintervals):
                fromfile = f'{homedir}rawpathweightfiles/pathweight_dring_sw{sw}_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{intervaldict[interval]}.var'
                tofile = f'{homedir}experiments/nsdi26fall/eval_scale_larger_supernode/pwfiles/pathweight_dring_sw{sw}_su2_prv1_load{load}_interval{interval}.pw'
                fcopy.write(f'cp {fromfile} {tofile}\n')

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/eval_scale_larger_supernode/)
python3 ../../../pararun.py --conf dringsu2_generate_pwfiles.conf --worker 40
+ actually copy dringsu2_copy_pwfiles.conf

In [23]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_scale_larger_supernode/run_dringsu2.conf'
with open(conffile, 'w') as frun:
    for isw,sw in enumerate(swlist):
        # set parameters
        topologyfile = f"evalscaletopologyfiles/dring_{sw}_{int(sw*0.8)}.edgelist"
        serverfile = f"evalserverfiles/dring_{dringserverlist[isw]}_{sw}_{int(sw*0.8)}.sv"
        npfile = f"evalscalenetpathfiles/netpath_dring_{sw}_{int(sw*0.8)}_su2.np"
        nhosts = dringserverlist[isw]
        nswitches = sw
        k = int(sw*0.8)
        with open(f"{homedir}{topologyfile}", 'r') as f:
            nlines = sum(1 for _ in f)
        nlinks = nlines * 2





    
        for seed in seed_list:
            cmfile = f'experiments/nsdi26fall/eval_scale_larger_supernode/cmfiles/dring_sw{sw}_load{load}.cm'
            pwfileprefix = f'experiments/nsdi26fall/eval_scale_larger_supernode/pwfiles/pathweight_dring_sw{sw}_su2_prv1_load{load}_interval'
            outfile = f'experiments/nsdi26fall/eval_scale_larger_supernode/outfiles/dringsu2_sw{sw}_seed{seed}.out'
            frun.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

python3 pararun.py --conf experiments/nsdi26fall/eval_scale_larger_supernode/run_dringsu2.conf --worker 25